In [1]:
import numpy as np
import cv2
import os
import hashlib
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

In [2]:
IMG_SIZE = 128

def load_data(path):
    data = []
    labels = []
    
    for label, folder in enumerate(["real", "fake"]):
        folder_path = os.path.join(path, folder)
        
        for img_name in os.listdir(folder_path):
            img_path = os.path.join(folder_path, img_name)
            
            img = cv2.imread(img_path)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img = img / 255.0
            
            data.append(img)
            labels.append(label)
    
    return np.array(data), np.array(labels)

X, y = load_data("dataset/")
print("Data Loaded:", X.shape)

Data Loaded: (2041, 128, 128, 3)


In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [4]:
model = Sequential()

model.add(Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)))
model.add(MaxPooling2D(2,2))

model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D(2,2))

model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.summary()

C:\Users\gudar\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 126, 126, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 63, 63, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 61, 61, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 30, 30, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 57600)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │       7,372,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 7,392,449 (28.20 MB)

 Trainable params: 7,392,449 (28.20 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

Epoch 1/5
51/51 ━━━━━━━━━━━━━━━━━━━━ 18s 312ms/step - accuracy: 0.5294 - loss: 0.8865 - val_accuracy: 0.5330 - val_loss: 0.6878
Epoch 2/5
51/51 ━━━━━━━━━━━━━━━━━━━━ 20s 306ms/step - accuracy: 0.5754 - loss: 0.6796 - val_accuracy: 0.5306 - val_loss: 0.6901
Epoch 3/5
51/51 ━━━━━━━━━━━━━━━━━━━━ 15s 300ms/step - accuracy: 0.6250 - loss: 0.6538 - val_accuracy: 0.5795 - val_loss: 0.6844
Epoch 4/5
51/51 ━━━━━━━━━━━━━━━━━━━━ 21s 314ms/step - accuracy: 0.6875 - loss: 0.6132 - val_accuracy: 0.5795 - val_loss: 0.6877
Epoch 5/5
51/51 ━━━━━━━━━━━━━━━━━━━━ 16s 313ms/step - accuracy: 0.7114 - loss: 0.5655 - val_accuracy: 0.5379 - val_loss: 0.7328


In [6]:
# Watermark
def add_watermark(img):
    cv2.putText(img, "SECURE", (20,40),
                cv2.FONT_HERSHEY_SIMPLEX, 1,
                (0,255,0), 2)
    return img

# Hash
def generate_hash(image_path):
    with open(image_path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

# Blockchain (simple list)
blockchain = []

def add_block(data):
    blockchain.append(data)

In [7]:
def predict_image(image_path):
    
    img = cv2.imread(image_path)

    if img is None:
        print("Image not found ❌")
        return

    img_resized = cv2.resize(img, (128,128))
    img_resized = img_resized / 255.0
    img_resized = img_resized.reshape(1,128,128,3)

    pred = model.predict(img_resized)

    if pred[0][0] > 0.5:
        print("Fake Image ❌")
    else:
        print("Real Image ✅")

        # Security part
        img = add_watermark(img)

        h = generate_hash(image_path)
        print("Hash:", h)

        add_block(h)
        print("Stored in Blockchain")

In [11]:
predict_image("dataset/real/real_01024.jpg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
Real Image ✅
Hash: d6344d39e9f988e8df4318dd470bdf5faa655819d128816b6732d6c083f699e6
Stored in Blockchain


In [13]:
predict_image("dataset/fake/mid_5_0011.jpg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
Fake Image ❌
